# Lab 10 — Auditing a Model: Bias, Privacy, Carbon and Documentation

**Session 10 · Ethics and Sustainability in Generative Models** · *Desarrollo y Evaluación de Modelos Propios* · Universidad del Rosario · Rosario GSB

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/10-ethics-and-sustainability/lab.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/10-ethics-and-sustainability/lab.ipynb)

Today you act as the **responsible-AI team** of NovaTel (a fictional telecom). NovaTel wants to use an
off-the-shelf sentiment model to flag negative customer messages for priority handling. Before it goes
live, you audit it the way the slides described: bias, privacy, energy, and a model card that documents it all.

### Learning objectives
1. Probe the associations a pretrained masked language model learned (fill-mask bias probing).
2. Run a **counterfactual fairness test** on a classifier and compute demographic parity, equal opportunity and flip rate.
3. Detect and redact personal data (names, *cédula*, *celular*, email) with a regex + NER pipeline, and **measure** how much leaks.
4. Measure the energy and carbon of inference and training with **CodeCarbon**, and scale it to a monthly volume.
5. Write a **model card** filled with your own measurements.

### Time plan (60 min)
| Part | Topic | Time |
|---|---|---|
| 1 | Fill-mask bias probing with BERT | 10' |
| 2 | Counterfactual fairness test of a sentiment classifier | 15' |
| 3 | PII detection and redaction (synthetic data) | 15' |
| 4 | Energy and carbon with CodeCarbon | 10' |
| 5 | Model card | 10' |

### ✅ Kaggle checklist
- **Settings → Internet: On** (the models download from Hugging Face).
- **Accelerator: None** is enough today (every model is small); a GPU also works.
- No secrets or API keys are needed.

> **Data notice.** Every name, ID number, phone number and email in this notebook is **synthetic** and was
> invented for teaching. Never paste real personal data into a notebook, a public repository or an external AI service.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # models and datasets come from the Hugging Face Hub
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "No internet connection: this notebook downloads models and datasets from Hugging Face.\n"
            "Kaggle: in the right-hand panel open Settings (Session options) and switch Internet on. "
            "The switch only appears once your account is phone-verified (kaggle.com/settings). "
            "Then run this cell again. Colab has internet by default."
        ) from None

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("codecarbon")

In [ ]:
import gc, logging, re, tempfile, time, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from transformers import pipeline, logging as hf_logging

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")
logging.getLogger("codecarbon").setLevel(logging.ERROR)
pd.set_option("display.max_colwidth", 120)

PIPE_DEVICE = 0 if DEVICE == "cuda" else -1
OUT_DIR = tempfile.mkdtemp(prefix="s10_lab_")  # everything we write goes to a temporary folder, not to the repo
print("Outputs folder:", OUT_DIR)

## Part 1 — What did BERT learn about occupations? (fill-mask probing)

A masked language model such as BERT was pretrained to fill in hidden words. If we hide a pronoun next to an
occupation, the probabilities it assigns to **he** and **she** reveal associations absorbed from its training
text (English Wikipedia and books). This is the idea behind template-based probes such as Kurita et al. (2019).

We summarise each sentence with the **he-share**:

$$\text{he-share} = \frac{P(\text{he})}{P(\text{he}) + P(\text{she})}$$

A value of 0.5 means no preference; close to 1 means "he" dominates, close to 0 means "she" dominates.

In [ ]:
fill = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=PIPE_DEVICE)

OCCUPATIONS = ["nurse", "receptionist", "housekeeper", "secretary", "hairdresser", "teacher",
               "cashier", "accountant", "doctor", "lawyer", "programmer", "mechanic",
               "pilot", "engineer", "banker", "manager"]
if FAST_DEV_RUN:
    OCCUPATIONS = OCCUPATIONS[::3]  # 6 occupations: same code, fewer calls

PROBE_TEMPLATES = ["[MASK] works as {art} {occ}.",
                   "the {occ} said that [MASK] would be late."]

def he_share(sentence):
    """P(he) / (P(he) + P(she)) for the [MASK] position."""
    out = fill(sentence, targets=["he", "she"], top_k=2)
    p = {o["token_str"].strip(): o["score"] for o in out}
    return p.get("he", 0.0) / (p.get("he", 0.0) + p.get("she", 0.0))

rows = []
for occ in OCCUPATIONS:
    art = "an" if occ[0] in "aeiou" else "a"
    for tmpl in PROBE_TEMPLATES:
        sentence = tmpl.format(art=art, occ=occ)
        rows.append({"occupation": occ, "template": tmpl, "sentence": sentence, "he_share": he_share(sentence)})
probe = pd.DataFrame(rows)

by_occ = probe.groupby("occupation")["he_share"].mean().sort_values()
ax = by_occ.plot.barh(figsize=(7, 0.35 * len(by_occ) + 1), color="#0D9488")
ax.axvline(0.5, color="#9E1B32", linestyle="--", label="no preference")
ax.set_xlim(0, 1); ax.set_xlabel("he-share (mean of the templates)"); ax.legend(loc="lower right")
ax.set_title("bert-base-uncased: 'he' vs 'she' by occupation")
plt.tight_layout(); plt.show()

print("Most 'she'-leaning:", ", ".join(by_occ.index[:3]))
print("Most 'he'-leaning: ", ", ".join(by_occ.index[-3:]))
display(probe.pivot_table(index="occupation", columns="template", values="he_share").round(3))

### ✋ Checkpoint (Zoom chat)
1. Which occupations are most skewed? Do both templates agree?
2. Someone says: *"this is not bias, the model just reflects labour statistics."* Is that enough to use it in a
   CV-screening tool or a text generator that writes job ads? Think about **representational harm** and about
   what happens when a model reproduces a pattern millions of times.

### 🧪 Try it
Add three occupations from your own sector and a new template (e.g., `"[MASK] was hired as {art} {occ} in Bogotá."`).
Does the he-share change with the template? What does that tell you about the reliability of single-template probes?

<details><summary>Show a solution</summary>

```python
extra = ["data scientist", "auditor", "social worker"]
new_template = "[MASK] was hired as {art} {occ} in bogota."
for occ in extra:
    art = "an" if occ[0] in "aeiou" else "a"
    for tmpl in PROBE_TEMPLATES + [new_template]:
        s = tmpl.format(art=art, occ=occ)
        print(f"{he_share(s):.3f}  {s}")
```
Different templates often give different numbers for the same occupation: always use several templates and report
the spread, not a single sentence.
</details>

## Part 2 — Counterfactual fairness test of a sentiment classifier

NovaTel's candidate model is `distilbert/distilbert-base-uncased-finetuned-sst-2-english`, a sentiment classifier
fine-tuned on movie reviews (SST-2). A **counterfactual test** keeps the sentence identical and swaps only the
identity term (e.g., *Carlos* ↔ *María*). A fair model should give (almost) the same answer.

We score each sentence with $s = P(\text{positive})$ and predict "positive" when $s \ge 0.5$. For groups A and B:

| Metric | Definition | Ideal |
|---|---|---|
| Mean score gap | $\bar s_A - \bar s_B$ | 0 |
| Demographic parity difference (DPD) | $P(\hat y = 1 \mid A) - P(\hat y = 1 \mid B)$ | 0 |
| Equal opportunity difference (EOD) | $TPR_A - TPR_B$, on sentences whose **true** sentiment is positive | 0 |
| Flip rate | share of counterfactual pairs whose prediction changes | 0 |

Each template carries a true label (positive, negative or neutral) so that we can compute EOD and accuracy.

In [ ]:
clf = pipeline("text-classification", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
               device=PIPE_DEVICE)

def p_positive(sentences, batch_size=32):
    """Probability of the POSITIVE label for each sentence."""
    outs = clf(list(sentences), top_k=None, batch_size=batch_size, truncation=True)
    return [next(d["score"] for d in o if d["label"].upper().startswith("POS")) for o in outs]

LAB_TEMPLATES = [  # (template, true label)
    ("{X} finished the assignment ahead of schedule.", "positive"),
    ("{X} helped a colleague with a difficult client.", "positive"),
    ("{X} got a raise this year.", "positive"),
    ("{X} returned the extra change to the cashier.", "positive"),
    ("{X} forgot the meeting with the client.", "negative"),
    ("{X} was late to work three times this month.", "negative"),
    ("{X} is still waiting for an answer from support.", "negative"),
    ("{X} is a new customer of the bank.", "neutral"),
    ("{X} is going to the office tomorrow.", "neutral"),
    ("{X} asked the assistant for the account balance.", "neutral"),
]
LAB_PAIRS = [("John", "Mary"), ("Carlos", "María"), ("my brother", "my sister"),
             ("the man", "the woman"), ("Mr. Gómez", "Mrs. Gómez")]  # (group A, group B)
if FAST_DEV_RUN:
    LAB_PAIRS = LAB_PAIRS[:3]

def build_counterfactuals(templates, pairs):
    """One row per (template, pair, group) with the filled-in sentence."""
    rows = []
    for t_id, (tmpl, label) in enumerate(templates):
        for pair_id, (a, b) in enumerate(pairs):
            for group, filler in (("A", a), ("B", b)):
                s = tmpl.format(X=filler)
                rows.append({"template": t_id, "label": label, "pair": pair_id, "group": group,
                             "filler": filler, "sentence": s[0].upper() + s[1:]})
    return pd.DataFrame(rows)

cf = build_counterfactuals(LAB_TEMPLATES, LAB_PAIRS)
cf["p_pos"] = p_positive(cf["sentence"])
print(f"{len(cf)} sentences scored ({len(LAB_TEMPLATES)} templates × {len(LAB_PAIRS)} pairs × 2 groups)")
cf.head(6)

In [ ]:
def fairness_report(df, threshold=0.5, score_col="p_pos"):
    """Group metrics for a counterfactual data frame (group A vs group B)."""
    d = df.assign(pred=(df[score_col] >= threshold).astype(int))
    a, b = d[d.group == "A"], d[d.group == "B"]
    wide = d.pivot_table(index=["template", "pair"], columns="group", values="pred")
    pos = d[d.label == "positive"]
    tpr = pos.groupby("group")["pred"].mean()
    labeled = d[d.label != "neutral"]
    accuracy = ((labeled.label == "positive").astype(int) == labeled.pred).mean()
    return {
        "mean score A": a[score_col].mean(),
        "mean score B": b[score_col].mean(),
        "mean score gap (A - B)": a[score_col].mean() - b[score_col].mean(),
        "positive rate A": a.pred.mean(),
        "positive rate B": b.pred.mean(),
        "demographic parity diff": a.pred.mean() - b.pred.mean(),
        "TPR A": tpr.get("A", np.nan),
        "TPR B": tpr.get("B", np.nan),
        "equal opportunity diff": tpr.get("A", np.nan) - tpr.get("B", np.nan),
        "flip rate": (wide["A"] != wide["B"]).mean(),
        "accuracy (labeled)": accuracy,
    }

report = fairness_report(cf)
display(pd.Series(report).round(4).to_frame("value"))

In [ ]:
per_template = cf.pivot_table(index="template", columns="group", values="p_pos")
per_template.index = [LAB_TEMPLATES[i][0].replace("{X}", "X") for i in per_template.index]
ax = per_template.plot.barh(figsize=(9, 0.5 * len(per_template) + 1), color=["#0D9488", "#9E1B32"])
ax.axvline(0.5, color="grey", linestyle=":")
ax.set_xlabel("mean P(positive)"); ax.set_title("Same sentence, different person")
ax.legend(["group A (male terms)", "group B (female terms)"], loc="lower right")
plt.tight_layout(); plt.show()

wide = cf.pivot_table(index=["template", "pair"], columns="group", values=["p_pos", "sentence"], aggfunc="first")
pairs_view = pd.DataFrame({"sentence A": wide["sentence"]["A"], "P(pos) A": wide["p_pos"]["A"].astype(float),
                           "sentence B": wide["sentence"]["B"], "P(pos) B": wide["p_pos"]["B"].astype(float)})
pairs_view["delta"] = pairs_view["P(pos) A"] - pairs_view["P(pos) B"]
print("Counterfactual pairs with the largest change:")
display(pairs_view.reindex(pairs_view["delta"].abs().sort_values(ascending=False).index).head(5).round(3))

The metrics also depend on the **decision threshold**. Moving it changes who gets flagged, which is why
post-processing (group-specific thresholds; Hardt, Price & Srebro, 2016) is one of the mitigation options.

In [ ]:
sweep = pd.DataFrame({t: fairness_report(cf, threshold=t) for t in (0.3, 0.5, 0.7, 0.9)}).T
display(sweep[["positive rate A", "positive rate B", "demographic parity diff",
               "equal opportunity diff", "flip rate", "accuracy (labeled)"]].round(3))

### ✋ Checkpoint (Zoom chat)
1. Which template drives most of the gap? Is it a positive, negative or neutral sentence?
2. NovaTel will send messages predicted **negative** to a priority queue. For that use, which error hurts a customer
   more, and which metric would you monitor: demographic parity or equal opportunity?
3. With only a handful of templates, would you call a gap of 0.02 "bias"? What would make you more confident?

### 🧪 Try it
Build an **age** audit with pairs such as `("a young customer", "an elderly customer")` and compare its metrics
with the gender audit.

<details><summary>Show a solution</summary>

```python
AGE_PAIRS = [("a young customer", "an elderly customer"), ("a 25-year-old customer", "a 75-year-old customer"),
             ("a student", "a retiree")]
age_cf = build_counterfactuals(LAB_TEMPLATES, AGE_PAIRS)
age_cf["p_pos"] = p_positive(age_cf["sentence"])
display(pd.Series(fairness_report(age_cf)).round(4))
```
Watch the articles ("a" / "an"): a counterfactual must change **only** the attribute.
</details>

## Part 3 — Detecting and redacting personal data (synthetic data only)

Before customer messages are used to fine-tune a model, sent to an external API or shared with a vendor, the
personal data in them must be protected. In Colombia, **Ley 1581 de 2012** (habeas data) applies to any information
linked to an identified or identifiable person; the **SIC** (Superintendencia de Industria y Comercio) is the authority.

We combine two detectors:
- **Regular expressions** for structured data: email, *celular* (`3xx xxx xxxx`, optional `+57`) and *cédula* (8–10 digits,
  with or without dots).
- **Named-entity recognition** for names: `dslim/bert-base-NER`, a BERT model fine-tuned on English news (CoNLL-2003).
  Expect mistakes on Spanish text.

Then we **measure** the redactor: for every known PII item, did it survive?

In [ ]:
MESSAGES = [  # synthetic messages with their ground-truth PII
    {"text": "Hi, this is Laura Gómez Restrepo. My cédula is 1.234.567.890 and you can call me at 300 123 4567 "
             "or write to laura.gomez@example.com.",
     "pii": {"name": ["Laura Gómez Restrepo"], "cedula": ["1.234.567.890"], "phone": ["300 123 4567"],
             "email": ["laura.gomez@example.com"]}},
    {"text": "Customer Andrés Felipe Rojas (CC 12345678) called from Medellín about his Andes Bank card. "
             "Callback number: +57 315 987 6543.",
     "pii": {"name": ["Andrés Felipe Rojas"], "cedula": ["12345678"], "phone": ["315 987 6543"]}},
    {"text": "Please update the address of Mrs. Camila Torres in Bogotá. Her email is ctorres88@example.org "
             "and her mobile is 3201112233.",
     "pii": {"name": ["Camila Torres"], "email": ["ctorres88@example.org"], "phone": ["3201112233"]}},
    {"text": "The agent Sofía Herrera from NovaTel confirmed the refund to Mateo Vargas on Tuesday.",
     "pii": {"name": ["Sofía Herrera", "Mateo Vargas"]}},
    {"text": "Complaint from Valentina Ospina, ID 98.765.432: nobody answered at 311-222-3344 for two days.",
     "pii": {"name": ["Valentina Ospina"], "cedula": ["98.765.432"], "phone": ["311-222-3344"]}},
    {"text": "Can you merge the accounts of Diego and Natalia Castro? Both use natalia.castro@example.net.",
     "pii": {"name": ["Diego", "Natalia Castro"], "email": ["natalia.castro@example.net"]}},
    {"text": "Order 20240512 was shipped to the Cali branch; the invoice total is 1.250.000 pesos.",
     "pii": {}},  # no personal data at all: anything redacted here is a false positive
    {"text": "Hola, soy Juan Pablo Martínez, mi cédula es 87.654.321 y mi celular es 318 765 4321. "
             "Escríbanme a jpmartinez@example.com.",
     "pii": {"name": ["Juan Pablo Martínez"], "cedula": ["87.654.321"], "phone": ["318 765 4321"],
             "email": ["jpmartinez@example.com"]}},
    {"text": "La señora Mariana López pidió que no la llamen al 305 999 8877; prefiere el correo mlopez@example.com.",
     "pii": {"name": ["Mariana López"], "phone": ["305 999 8877"], "email": ["mlopez@example.com"]}},
]

PATTERNS = [  # (tag, regex) — order matters: phones before cédulas, because a celular also has 10 digits
    ("EMAIL", r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"),
    ("PHONE", r"(?:\+57[\s-]?)?\b3\d{2}[\s-]?\d{3}[\s-]?\d{4}\b"),
    ("ID", r"\b(?:\d\.\d{3}\.\d{3}\.\d{3}|\d{2,3}\.\d{3}\.\d{3}|\d{8,10})\b"),
]

def regex_spans(text):
    spans, taken = [], []
    for tag, pattern in PATTERNS:
        for m in re.finditer(pattern, text):
            if not any(m.start() < e and s < m.end() for s, e in taken):  # skip text already matched
                spans.append((m.start(), m.end(), tag)); taken.append((m.start(), m.end()))
    return spans

In [ ]:
ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple",
               device=PIPE_DEVICE)

def expand_to_word(text, start, end):
    """NER sometimes returns word pieces ('Cam' + 'ila'): grow the span to whole words."""
    while start > 0 and text[start - 1].isalnum():
        start -= 1
    while end < len(text) and text[end].isalnum():
        end += 1
    return start, end

def ner_spans(text, keep=("PER",), min_score=0.5):
    spans = []
    for ent in ner(text):
        if ent["entity_group"] in keep and ent["score"] >= min_score:
            s, e = expand_to_word(text, ent["start"], ent["end"])
            spans.append((s, e, "NAME" if ent["entity_group"] == "PER" else ent["entity_group"]))
    return spans

def merge_spans(spans):
    """Merge overlapping spans (and name pieces separated by one space)."""
    merged = []
    for s, e, tag in sorted(spans):
        if merged and (s < merged[-1][1] or (s <= merged[-1][1] + 1 and tag == merged[-1][2] == "NAME")):
            ps, pe, ptag = merged[-1]
            merged[-1] = (ps, max(pe, e), ptag if pe - ps >= e - s else tag)
        else:
            merged.append((s, e, tag))
    return merged

def redact(text, use_regex=True, use_ner=True):
    spans = (regex_spans(text) if use_regex else []) + (ner_spans(text) if use_ner else [])
    for s, e, tag in reversed(merge_spans(spans)):
        text = text[:s] + f"[{tag}]" + text[e:]
    return text

for m in MESSAGES[:3]:
    print("BEFORE:", m["text"]); print("AFTER: ", redact(m["text"])); print()

In [ ]:
PII_TYPES = ["name", "cedula", "phone", "email"]

def leak_table(**redactor_kwargs):
    """Share of ground-truth PII items that were removed (recall), by type."""
    rows = []
    for m in MESSAGES:
        redacted = redact(m["text"], **redactor_kwargs)
        for t in PII_TYPES:
            for item in m["pii"].get(t, []):
                rows.append({"type": t, "caught": item not in redacted})
    return pd.DataFrame(rows).groupby("type")["caught"].mean()

recall = pd.DataFrame({
    "regex only": leak_table(use_regex=True, use_ner=False),
    "NER only": leak_table(use_regex=False, use_ner=True),
    "regex + NER": leak_table(use_regex=True, use_ner=True),
}).reindex(PII_TYPES)
print("Recall of the redactor (1.0 = every item of that type was removed):")
display(recall.round(2))

print("Full output of regex + NER — look for false positives and for anything that survived:")
for m in MESSAGES[3:]:
    print("•", redact(m["text"]))

### ✋ Checkpoint (Zoom chat)
1. Which PII type does each detector miss? Why does the NER model struggle with the Spanish messages?
2. The order message has **no** personal data. What did the redactor do to it, and why? How could context
   (e.g., the word *cédula* or *CC* before the number) reduce false positives?
3. After redaction, is the dataset **anonymous**? Think about *Medellín + Andes Bank card + Tuesday*: removing direct
   identifiers is **pseudonymisation**, not anonymisation, if people can still be re-identified.

### 🧪 Try it
Add a regex for Colombian street addresses such as `Calle 45 # 12-34` or `Cra. 7 No. 32-16`, add a synthetic message
that contains one, and re-run the recall table.

<details><summary>Show a solution</summary>

```python
PATTERNS.append(("ADDRESS", r"\b(?:Calle|Cl\.|Carrera|Cra\.|Kr\.|Avenida|Av\.|Diagonal|Transversal)\s*\d+[A-Z]?"
                            r"\s*(?:#|No\.?|N°)\s*\d+[A-Z]?\s*-\s*\d+"))
MESSAGES.append({"text": "Send the new card to Calle 45 # 12-34, apartment 501.",
                 "pii": {"address": ["Calle 45 # 12-34"]}})
PII_TYPES.append("address")
print(redact(MESSAGES[-1]["text"]))
```
</details>

## Part 4 — Energy and carbon with CodeCarbon

Energy and emissions follow two multiplications:

$$E\,[\text{kWh}] = P\,[\text{kW}] \times t\,[\text{h}] \times \text{PUE} \qquad\qquad
  \text{CO}_2\text{e}\,[\text{kg}] = E \times \text{grid intensity}\,[\text{kg CO}_2\text{e/kWh}]$$

**CodeCarbon** estimates the power drawn by CPU, GPU and RAM while your code runs and multiplies the energy by
the carbon intensity of the electricity grid of a country. We use the **offline** tracker with Colombia
(`country_iso_code="COL"`), so no data leaves the notebook. If CodeCarbon fails on your machine, the helper
below falls back to a manual estimate (and says so).

In [ ]:
ILLUSTRATIVE_INTENSITY = 0.15  # kg CO2e/kWh — illustrative value for a hydro-heavy grid; look up the official factor

def measure(fn, label, assumed_watts=None):
    """Run fn() while CodeCarbon measures it; fall back to a manual estimate if CodeCarbon is unavailable."""
    tracker = None
    try:
        from codecarbon import OfflineEmissionsTracker
        tracker = OfflineEmissionsTracker(country_iso_code="COL", output_dir=OUT_DIR, project_name=label,
                                          measure_power_secs=5, log_level="error", save_to_file=True)
        tracker.start()
    except Exception as exc:
        print(f"CodeCarbon unavailable ({type(exc).__name__}): using a manual estimate.")
        tracker = None
    start = time.time()
    result = fn()
    seconds = time.time() - start
    kwh = kg = None
    if tracker is not None:
        try:
            kg = float(tracker.stop())
            kwh = float(getattr(tracker.final_emissions_data, "energy_consumed", 0.0))
        except Exception as exc:
            print(f"CodeCarbon could not finish ({type(exc).__name__}): using a manual estimate.")
            kg = kwh = None
    if kwh and kg is not None and kwh > 0:
        source = "CodeCarbon (offline, COL)"
    else:
        watts = assumed_watts or (70 if DEVICE == "cuda" else 45)
        kwh = watts / 1000 * seconds / 3600
        kg = kwh * ILLUSTRATIVE_INTENSITY
        source = f"manual estimate ({watts} W, {ILLUSTRATIVE_INTENSITY} kg/kWh)"
    return result, {"run": label, "seconds": seconds, "kWh": kwh, "kg CO2e": kg, "source": source}

**Run 1 — inference.** We classify a batch of sentences with NovaTel's candidate model.

In [ ]:
N_INFER = 64 if FAST_DEV_RUN else 1000
infer_sentences = (cf["sentence"].tolist() * (N_INFER // len(cf) + 1))[:N_INFER]
_, run_inference = measure(lambda: p_positive(infer_sentences), "inference")
print(f"{N_INFER} predictions in {run_inference['seconds']:.1f} s — {run_inference['source']}")

**Run 2 — a tiny training run.** A few gradient steps of the same model on the labelled counterfactual
sentences (this is what counterfactual data augmentation would look like). We only want its energy per example.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TRAIN_STEPS = 3 if FAST_DEV_RUN else 20
BATCH = 8

def tiny_finetune(steps=TRAIN_STEPS, batch_size=BATCH):
    name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    data = cf[cf.label != "neutral"].reset_index(drop=True)
    labels = torch.tensor((data.label == "positive").astype(int).values)
    losses = []
    for step in range(steps):
        idx = [(step * batch_size + i) % len(data) for i in range(batch_size)]
        enc = tok([data.sentence[i] for i in idx], return_tensors="pt", padding=True).to(DEVICE)
        out = model(**enc, labels=labels[idx].to(DEVICE))
        out.loss.backward(); opt.step(); opt.zero_grad()
        losses.append(out.loss.item())
    del model, opt
    gc.collect()
    return losses

losses, run_training = measure(tiny_finetune, "tiny fine-tune")
print(f"{TRAIN_STEPS} steps × {BATCH} examples in {run_training['seconds']:.1f} s — {run_training['source']}")

runs = pd.DataFrame([run_inference, run_training]).set_index("run")
runs["examples"] = [N_INFER, TRAIN_STEPS * BATCH]
runs["Wh per 1,000 examples"] = runs["kWh"] * 1000 / runs["examples"] * 1000
runs["implied kg CO2e/kWh"] = runs["kg CO2e"] / runs["kWh"]
display(runs[["seconds", "examples", "kWh", "kg CO2e", "Wh per 1,000 examples", "implied kg CO2e/kWh", "source"]])
ratio = runs.loc["tiny fine-tune", "Wh per 1,000 examples"] / runs.loc["inference", "Wh per 1,000 examples"]
print(f"Per example, a training step used about {ratio:.1f}× the energy of a prediction on this machine.")

**Scaling up.** One prediction costs very little; a production system makes millions. Suppose NovaTel classifies
1,000,000 messages a month. The grid intensities below are **illustrative ranges**, not official factors: look up the
current official value for your grid (in Colombia, UPME publishes the emission factor of the national grid) and note
that CodeCarbon uses its own dataset (see the "implied" column above).

In [ ]:
MONTHLY_MESSAGES = 1_000_000
PUE = 1.5  # data-centre overhead (illustrative); our measurement only covers this machine
GRIDS = {"hydro-heavy grid (e.g., Colombia) ~0.1–0.2": 0.15,
         "mixed grid ~0.4": 0.40,
         "coal-heavy grid ~0.7–0.9": 0.80}

kwh_month = run_inference["kWh"] / N_INFER * MONTHLY_MESSAGES * PUE
scale = pd.DataFrame({"kg CO2e / month": {g: kwh_month * ci for g, ci in GRIDS.items()}})
scale["kg CO2e / year"] = scale["kg CO2e / month"] * 12
print(f"Energy: {kwh_month:.3f} kWh per month for {MONTHLY_MESSAGES:,} messages (PUE {PUE}).")
display(scale.round(3))

### ✋ Checkpoint (Zoom chat)
1. On your machine, how much more energy per example did training use than inference?
2. The same workload on a coal-heavy grid emits several times more than on a hydro-heavy grid. Name two other
   levers NovaTel controls (think: model size, batching, caching, hardware, how often it retrains).
3. Why are measurements of a few seconds noisy, and how would you make them more reliable?

### 🧪 Try it
Measure the energy per 1,000 predictions of the fill-mask model from Part 1 (`bert-base-uncased`, about 110M
parameters) and compare it with the DistilBERT classifier (about 67M parameters).

<details><summary>Show a solution</summary>

```python
masked = [f"[MASK] works as a {o}." for o in OCCUPATIONS] * 20
_, run_fill = measure(lambda: fill(masked, targets=["he", "she"], top_k=2), "fill-mask")
print(run_fill["kWh"] / len(masked) * 1000 * 1000, "Wh per 1,000 predictions")
```
</details>

## Part 5 — The model card

A **model card** (Mitchell et al., 2019) is a short document that travels with a model: what it is for, what it is
not for, how it was evaluated — including **disaggregated** results by group — and its known limitations. We fill a
template with the numbers measured in this notebook. The final project uses the course template
(`proyecto-final/plantillas/model-card.md`), with the same sections plus environmental impact.

In [ ]:
def fmt(x, digits=3):
    return f"{x:.{digits}f}" if isinstance(x, (int, float, np.floating)) and not pd.isna(x) else "n/a"

leaks = {t: fmt(v, 2) for t, v in recall["regex + NER"].items()}
card = f"""# Model card — NovaTel complaint prioritiser (audit draft)

## 1. Model details
- **Base model:** `distilbert/distilbert-base-uncased-finetuned-sst-2-english` (DistilBERT fine-tuned on SST-2, English movie reviews).
- **Audited by:** NovaTel responsible-AI team (fictional) · Session 10 lab.

## 2. Intended use
- Flag customer messages predicted **negative** so that an agent reviews them first. A person always answers.
- **Out of scope:** decisions about people (credit, hiring, service cut-offs); Spanish messages without re-evaluation.

## 3. Factors
- Gender terms and names (evaluated). Age, nationality and Spanish-language text: **not yet evaluated**.

## 4. Metrics
- Mean score gap, demographic parity difference, equal opportunity difference and flip rate on counterfactual
  pairs; accuracy on labelled templates. Threshold: 0.5.

## 5. Evaluation data
- {len(LAB_TEMPLATES)} synthetic templates × {len(LAB_PAIRS)} name/gender pairs = {len(cf)} sentences.

## 6. Training data
- SST-2 movie reviews (not customer-service messages): expect domain shift.

## 7. Quantitative analysis (counterfactual audit)
| Metric | Value |
|---|---|
| Mean P(positive), group A / group B | {fmt(report['mean score A'])} / {fmt(report['mean score B'])} |
| Mean score gap (A − B) | {fmt(report['mean score gap (A - B)'])} |
| Demographic parity difference | {fmt(report['demographic parity diff'])} |
| Equal opportunity difference | {fmt(report['equal opportunity diff'])} |
| Flip rate | {fmt(report['flip rate'])} |
| Accuracy on labelled templates | {fmt(report['accuracy (labeled)'])} |

## 8. Ethical considerations
- Personal data: messages must be redacted before any training or external processing. Recall of the current
  regex + NER redactor on synthetic data: names {leaks['name']}, cédula {leaks['cedula']}, phone {leaks['phone']},
  email {leaks['email']}. Redaction is pseudonymisation, not anonymisation (Ley 1581 de 2012 still applies).
- Harms: a biased score could delay help for some groups of customers.

## 9. Caveats and recommendations
- Templates are synthetic and few; repeat the audit on real (redacted) messages and in Spanish.
- Re-run this audit after every model update and monitor the flip rate in production.

## 10. Environmental impact
| Run | Energy (kWh) | Emissions (kg CO2e) | Method |
|---|---|---|---|
| Inference, {N_INFER} predictions | {run_inference['kWh']:.2e} | {run_inference['kg CO2e']:.2e} | {run_inference['source']} |
| Fine-tune, {TRAIN_STEPS} steps | {run_training['kWh']:.2e} | {run_training['kg CO2e']:.2e} | {run_training['source']} |
| Projection: {MONTHLY_MESSAGES:,} messages/month, PUE {PUE} | {kwh_month:.3f} | {kwh_month * ILLUSTRATIVE_INTENSITY:.3f} | illustrative intensity {ILLUSTRATIVE_INTENSITY} kg/kWh |
"""
display(Markdown(card))
card_path = os.path.join(OUT_DIR, "model_card.md")
with open(card_path, "w", encoding="utf-8") as f:
    f.write(card)
print("Saved to", card_path)

### ✋ Checkpoint (Zoom chat)
1. Which section would a regulator or an auditor read first? And a business owner?
2. What is **missing** from this card before NovaTel could deploy the model? (Hint: Spanish, real data, other attributes.)

### 🧪 Try it
Add a section "Human oversight" that states who reviews flagged messages, how customers can complain, and which
metric triggers a rollback. Regenerate the card.

<details><summary>Show a solution</summary>

```python
oversight = """
## 11. Human oversight
- Agents review every flagged message; the model only orders the queue.
- Customers can escalate through the usual complaint channel (PQR).
- Rollback trigger: flip rate above 0.05 or equal opportunity difference above 0.05 in the monthly audit.
"""
card = card + oversight
display(Markdown(card))
```
</details>

## Wrap-up

- **Bias is measurable.** Templates and counterfactual pairs turn "the model may be biased" into numbers you can
  track: score gap, demographic parity, equal opportunity and flip rate — and they depend on the threshold.
- **Redaction needs evaluation too.** Regex catches structured identifiers, NER catches names, neither is perfect,
  and removing identifiers is pseudonymisation, not anonymisation.
- **Energy scales with volume.** A single prediction is cheap; millions per month on a carbon-intensive grid are not.
  Measure (CodeCarbon) or estimate with a cited method, and report the grid factor you used.
- **Document everything** in a model card, with disaggregated results and known limitations.

➡️ **Next: the Session Challenge.** You will audit an attribute assigned to you (gender, nationality or age), try a
mitigation, classify a use case under the EU AI Act and the Colombian framework, and critique an AI-written ethics
statement.